# IS 445 Homework: Exploring Professional Licenses in Illinois

**Student:** Yashna Yadav  
**Course:** IS 445  

For this assignment, I used the Illinois professional licensing dataset to make two visualizations using Python, Pandas, and Altair. I wanted to compare which license categories appear most often and see how their license statuses differ.

Both charts use the **same dataset**, which I read directly from a public URL. I saved the charts as JSON files so I can display them on my GitHub Pages website.


In [1]:
import pandas as pd
import altair as alt

DATA_URL = (
    'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/'
    'is445_data/main/licenses_fall2022.csv'
)

df = pd.read_csv(DATA_URL, low_memory=False)
print('Rows, columns:', df.shape)
print('Altair version:', alt.__version__)
print('Columns of interest:', df[['Description', 'License Status']].columns.tolist())


Rows, columns: (10000, 31)
Altair version: 6.3.0
Columns of interest: ['Description', 'License Status']


## Step 1: Prepare the data

I started by keeping the license descriptions and license statuses because those are the two fields I wanted to explore. I removed rows where either field was missing and cleaned up extra spaces so the category names would be consistent.

For the first chart, I counted the records in each license category and kept the **top ten**. For the second chart, I used those same ten categories and grouped the statuses into **Active**, **Not renewed**, and **Other** to make the comparisons easier to read.

The counts represent records in this dataset, not the number of people currently working in these jobs.


In [2]:
licenses = df[['Description', 'License Status']].dropna().copy()
licenses['Description'] = licenses['Description'].astype(str).str.strip()
licenses['License Status'] = licenses['License Status'].astype(str).str.strip().str.upper()
licenses = licenses[(licenses['Description'] != '') & (licenses['License Status'] != '')]

# Count how many records are in each category and keep the top ten.
bar_counts = (
    licenses.groupby('Description')
    .size().reset_index(name='Records')
    .sort_values('Records', ascending=False)
    .head(10)
)
top_categories = bar_counts['Description'].tolist()

# Group the different license statuses into three categories for Plot 2.
licenses['Status Group'] = licenses['License Status'].map({
    'ACTIVE': 'Active',
    'NOT RENEWED': 'Not renewed'
}).fillna('Other')

status_counts = (
    licenses[licenses['Description'].isin(top_categories)]
    .groupby(['Description', 'Status Group'])
    .size().reset_index(name='Records')
)

display(bar_counts)
display(status_counts.head())


,Description,Records
89,PERMANENT EMPLOYEE REGISTRATION,3711
48,LICENSED COSMETOLOGIST,2726
31,FIREARM CONTROL CARD,729
88,ORIGINAL FIREARMS TRAINING,367
72,LICENSED NAIL TECHNICIAN,302
51,LICENSED DENTIST,268
40,LICENSED BARBER,235
13,BCENT SALON/SHOP REGISTRATION,229
52,LICENSED DENTIST CONTROLLED SUBSTANCE,210
91,REGISTERED DENTAL HYGIENIST,189


,Description,Status Group,Records
0,BCENT SALON/SHOP REGISTRATION,Active,50
1,BCENT SALON/SHOP REGISTRATION,Not renewed,146
2,BCENT SALON/SHOP REGISTRATION,Other,33
3,FIREARM CONTROL CARD,Active,24
4,FIREARM CONTROL CARD,Not renewed,217


## Plot 1: Which license categories appear most often?

My first chart is a horizontal bar chart showing the ten license categories with the most records. I chose horizontal bars because many of the category names are long and would be harder to read along the bottom of a graph. I also sorted the bars by count so it is easier to see which categories appear most often.

The license descriptions are **nominal (N)**, and the record counts are **quantitative (Q)**. I used one blue color for all the bars because I am only comparing counts, not showing another category with color.


In [3]:
# Plot 1: compare the number of records for the top ten licenses.
chart1 = alt.Chart(bar_counts).mark_bar(color='#346B8A').encode(
    x=alt.X('Records:Q', title='Number of records in sample'),
    y=alt.Y('Description:N', sort=top_categories, title='License category'),
    tooltip=[
        alt.Tooltip('Description:N', title='License category'),
        alt.Tooltip('Records:Q', format=',', title='Records')
    ]
).properties(
    width=550,
    height=330,
    title='Most common professional license categories in the sample'
)
chart1


alt.Chart(...)

## Plot 2: How do the license statuses compare?

For the second chart, I wanted to compare the proportion of different license statuses within the same ten categories. I used a **100% stacked horizontal bar chart** because the categories have different total numbers of records. Looking at percentages makes the comparison clearer.

The license categories and status groups are **nominal (N)**, while the share of records is **quantitative (Q)**. I used blue for **Active**, gold for **Not renewed**, and gray for **Other** so the groups are easy to tell apart.

### Interactivity

I added a clickable legend to this chart. Clicking a status in the legend highlights it and fades the others, which makes it easier to compare that status across categories. You can also Shift-click to select more than one status. This gives viewers another way to explore the results besides zooming or panning.


In [4]:
# Plot 2: use a clickable legend to highlight a license status.
status_selection = alt.selection_point(
    fields=['Status Group'],
    bind='legend'
)

chart2 = alt.Chart(status_counts).mark_bar().encode(
    x=alt.X(
        'Records:Q',
        stack='normalize',
        axis=alt.Axis(format='%'),
        title='Share of sample license records'
    ),
    y=alt.Y('Description:N', sort=top_categories, title='License category'),
    color=alt.Color(
        'Status Group:N',
        scale=alt.Scale(
            domain=['Active', 'Not renewed', 'Other'],
            range=['#2677A5', '#D7A345', '#A9B3BC']
        ),
        title='License status (click to highlight)'
    ),
    opacity=alt.condition(status_selection, alt.value(1), alt.value(0.15)),
    tooltip=[
        alt.Tooltip('Description:N', title='License category'),
        alt.Tooltip('Status Group:N', title='Status'),
        alt.Tooltip('Records:Q', title='Records', format=',')
    ]
).add_params(status_selection).properties(
    width=550,
    height=330,
    title='License status composition within each category'
)
chart2


alt.Chart(...)

## Step 2: Save the charts for my website

The code below saves both charts as `chart1.json` and `chart2.json`. These files are what my Jekyll webpage will use to display the visualizations, including the clickable legend in Plot 2.

After running this cell, I will upload both JSON files to `assets/plots/` in my GitHub Pages repository and upload this notebook to `python_notebooks/`.


In [5]:
chart1.save('chart1.json')
chart2.save('chart2.json')
print('Saved chart1.json and chart2.json. Upload both to assets/plots/ in your website repository.')


Saved chart1.json and chart2.json. Upload both to assets/plots/ in your website repository.
